# Assignment 03: Custom Modules (100 points)

**Unit 06: Programming PyTorch | AI 310**

In USAAIO Round 2, you must build neural network components from scratch using `nn.Module`. No high-level wrappers allowed. This assignment trains you to implement activation functions, custom layers, and compose them into complete architectures.

**Notation**:
- $B$ = batch size, $D$ = dimension
- All modules must support arbitrary batch sizes

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries. Do not use `nn.ReLU`, `nn.Sigmoid`, or `nn.GELU` — you must implement these from scratch.

---

## Part 1 (15 points, coding)

Implement ReLU as an `nn.Module`.

$$\text{ReLU}(x) = \max(0, x)$$

Your implementation must:
- Subclass `nn.Module`
- Implement `forward()` using only tensor operations
- Work for tensors of any shape
- Have zero learnable parameters

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyReLU(nn.Module):
    def __init__(self):
        pass

    def forward(self, x):
        # x: any shape
        # return: same shape, with ReLU applied element-wise
        pass

In [ ]:
""" END OF THIS PART """
relu = MyReLU()
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
assert torch.allclose(relu(x), torch.tensor([0.0, 0.0, 0.0, 1.0, 2.0]))
# Test gradient flow
x_grad = torch.randn(3, 4, requires_grad=True)
y = relu(x_grad).sum()
y.backward()
assert x_grad.grad is not None
assert sum(p.numel() for p in relu.parameters()) == 0
print("Part 1 passed!")

---

## Part 2 (15 points, coding)

Implement Sigmoid as an `nn.Module`.

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Do NOT use `torch.sigmoid`. Implement using `torch.exp` only.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MySigmoid(nn.Module):
    def __init__(self):
        pass
    
    def forward(self, x):
        pass

In [ ]:
""" END OF THIS PART """
sig = MySigmoid()
x = torch.tensor([-10.0, -1.0, 0.0, 1.0, 10.0])
expected = torch.sigmoid(x)
assert torch.allclose(sig(x), expected, atol=1e-5)
assert torch.allclose(sig(torch.tensor([0.0])), torch.tensor([0.5]))
print("Part 2 passed!")

---

## Part 3 (20 points, coding)

Implement a **Learnable Activation** module: the Swish activation with a learnable $\beta$ parameter.

$$\text{Swish}_{\beta}(x) = x \cdot \sigma(\beta x) = \frac{x}{1 + e^{-\beta x}}$$

Requirements:
- $\beta$ must be an `nn.Parameter` initialized to 1.0
- The optimizer must be able to update $\beta$ during training
- Use your `MySigmoid` from Part 2 (or `torch.sigmoid`)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class LearnableSwish(nn.Module):
    def __init__(self):
        pass
    
    def forward(self, x):
        pass

In [ ]:
""" END OF THIS PART """
swish = LearnableSwish()
assert sum(p.numel() for p in swish.parameters()) == 1, "Should have exactly 1 parameter (beta)"
x = torch.randn(5, 10, requires_grad=True)
y = swish(x).sum()
y.backward()
# Beta should receive a gradient
beta_param = list(swish.parameters())[0]
assert beta_param.grad is not None, "Beta must receive gradients"
# At beta=1, Swish(x) = x * sigmoid(x)
x_test = torch.tensor([0.0, 1.0, -1.0])
expected = x_test * torch.sigmoid(x_test)
assert torch.allclose(swish(x_test), expected, atol=1e-5)
print("Part 3 passed!")

---

## Part 4 (25 points, coding)

Implement a **custom Linear layer** from scratch. Do NOT use `nn.Linear`.

$$y = xW^T + b$$

Requirements:
- `weight` must be an `nn.Parameter` of shape `(out_features, in_features)`
- `bias` must be an `nn.Parameter` of shape `(out_features,)`
- Initialize weight from $\mathcal{U}(-\sqrt{k}, \sqrt{k})$ where $k = 1/\text{in\_features}$ (Kaiming uniform)
- Initialize bias from $\mathcal{U}(-\sqrt{k}, \sqrt{k})$
- Must produce the same result as `nn.Linear` for the same weights

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        pass
    
    def forward(self, x):
        # x: (B, in_features)
        # return: (B, out_features)
        pass

In [ ]:
""" END OF THIS PART """
my_linear = MyLinear(10, 5)
assert sum(p.numel() for p in my_linear.parameters()) == 10 * 5 + 5  # weight + bias
x = torch.randn(8, 10)
out = my_linear(x)
assert out.shape == (8, 5), f"Expected (8, 5), got {out.shape}"
# Verify equivalence with nn.Linear
ref = nn.Linear(10, 5)
ref.weight.data = my_linear.weight.data.clone()
ref.bias.data = my_linear.bias.data.clone()
assert torch.allclose(my_linear(x), ref(x), atol=1e-5)
# Gradient flow
x_g = torch.randn(4, 10, requires_grad=True)
my_linear(x_g).sum().backward()
assert x_g.grad is not None
print("Part 4 passed!")

---

## Part 5 (25 points, coding)

Build a complete **two-layer MLP** using ONLY your custom components from Parts 1-4.

Architecture:
- `MyLinear(in_dim, hidden_dim)` $\to$ `MyReLU()` $\to$ `MyLinear(hidden_dim, out_dim)`

Additionally, implement a `predict` method that returns class predictions (argmax of logits).

Do NOT use any `nn.Linear`, `nn.ReLU`, or `nn.Sequential`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        pass
    
    def forward(self, x):
        # x: (B, in_dim)
        # return: (B, out_dim) raw logits
        pass
    
    def predict(self, x):
        # x: (B, in_dim)
        # return: (B,) predicted class indices
        pass

In [ ]:
""" END OF THIS PART """
mlp = MyMLP(784, 256, 10)
total_params = sum(p.numel() for p in mlp.parameters())
expected_params = 784*256 + 256 + 256*10 + 10
assert total_params == expected_params, f"Expected {expected_params} params, got {total_params}"

x = torch.randn(32, 784)
logits = mlp(x)
assert logits.shape == (32, 10), f"Expected (32, 10), got {logits.shape}"

preds = mlp.predict(x)
assert preds.shape == (32,), f"Expected (32,), got {preds.shape}"
assert preds.dtype == torch.long or preds.dtype == torch.int64

# Check that no nn.Linear or nn.ReLU is used (inspect module tree)
for name, module in mlp.named_modules():
    assert not isinstance(module, nn.Linear), "Must use MyLinear, not nn.Linear"
    assert not isinstance(module, nn.ReLU), "Must use MyReLU, not nn.ReLU"

# Gradient flow
loss = nn.CrossEntropyLoss()(logits, torch.randint(0, 10, (32,)))
loss.backward()
all_have_grads = all(p.grad is not None for p in mlp.parameters())
assert all_have_grads, "All parameters must receive gradients"
print(f"Part 5 passed! Total parameters: {total_params:,}")